In [2]:
#import libraries
import re
import html
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
#put this csv inside a data folder.
file_path = Path("twitter_sample.csv")
if not file_path.exists():
    file_path = Path("data/twitter_sample.csv")
df = pd.read_csv(file_path)
print("Original rows:", len(df))
print(df["Tweet Type"].value_counts())

Original rows: 386
Tweet Type
ReTweet    276
Tweet       95
Reply       15
Name: count, dtype: int64


I loaded the CSV into pandas so I could work with the tweet text and filter the rows. I also printed the tweet types to see what was in the file. That helped me make sure I was using the right rows.

In [4]:
#create reusable class
class TweetTextProcessor:
    def __init__(self):
        self.stop_words = set(ENGLISH_STOP_WORDS)
        self.bow_vectorizer = CountVectorizer()
        self.tfidf_vectorizer = TfidfVectorizer()

    def clean_text(self, text):
        if pd.isna(text):
            return ""

        #decode HTML characters and lowercase the text
        text = html.unescape(str(text)).lower()

        #remove literal newline and tab markers.
        text = text.replace("\\n", " ")
        text = text.replace("\\r", " ")
        text = text.replace("\\t", " ")

        #remove URLs and usernames
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"@\w+", " ", text)

        #remove apostrophes
        text = re.sub(r"[’']", "", text)

        #keep alphabetic words with at least two letters
        words = re.findall(r"\b[a-z]{2,}\b", text)

        #remove english stop words
        cleaned_words = [
            word for word in words
            if word not in self.stop_words
        ]

        return " ".join(cleaned_words)

    def build_bow(self, documents):
        return self.bow_vectorizer.fit_transform(documents)

    def build_tfidf(self, documents):
        return self.tfidf_vectorizer.fit_transform(documents)

I created a class so I can reuse the same process for every tweet. The cleaning method removes links, usernames, numbers, punctuation,
and emojis. It also removes common English stop words and changes everything to lowercase. I kept the words in hashtags because they can provide useful information about the topic.

In [5]:
#clean tweet content cell
processor = TweetTextProcessor()

df["Cleaned Tweet"] = df["Tweet Content"].apply(
    processor.clean_text
)

print(
    df[["Tweet Content", "Cleaned Tweet"]]
    .head()
    .to_string(index=False)
)

                                                                                                                                                                                                                                             Tweet Content                                                                                                                                  Cleaned Tweet
Pets change our lives &amp; become a part of our families ❤️\nThat's why our members offer many solutions to help you to enjoy a long-lasting bond with your happy &amp; healthy pet 🐱🐶\n#MorethanMedicine #PetCare #PetsareFamily https://t.co/fZNIXge9a3 pets change lives families thats members offer solutions help enjoy long lasting bond happy healthy pet morethanmedicine petcare petsarefamily
                                                                   Another spot of our #morethanmedicine bus in #bristol this week! If you need support with your cancer diagnosis call us on 0303 3000 118. #living

I applied the cleaning method to the Tweet Content column and saved the results in a new column. I printed the original and cleaned text side by side so I could check what changed. Keeping the original text also lets me review details that the cleaning removed.

In [7]:
#keep original tweets only
tweet_filter = (
    df["Tweet Type"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("tweet")
)

tweets = df.loc[tweet_filter].copy()

#exclude any tweets that became empty after cleaning
tweets = tweets.loc[
    tweets["Cleaned Tweet"].str.strip().ne("")
].copy()

#keep the original CSV row index for identifying tweets
tweets = tweets.reset_index().rename(
    columns={"index": "CSV Row"}
)

documents = tweets["Cleaned Tweet"].tolist()

print("Original tweets available for analysis:", len(documents))
print(tweets[["CSV Row", "Cleaned Tweet"]].head().to_string(index=False))

Original tweets available for analysis: 95
 CSV Row                                                                                                                                  Cleaned Tweet
       0 pets change lives families thats members offer solutions help enjoy long lasting bond happy healthy pet morethanmedicine petcare petsarefamily
       1                                                      spot morethanmedicine bus bristol week need support cancer diagnosis livingwellwithcancer
       5                                                                                                                    great team morethanmedicine
      11                                      want preview story check monday changeofheart sat alin discuss new life heart transplant morethanmedicine
      12                                                     yall rose marie leslie blue ribbon winning vinaigrette maker morethanmedicine alltheskills


I kept only rows labeled Tweet and excluded retweets and replies. I also checked for tweets that became empty after cleaning.
All 95 original tweets were still usable. I saved the original CSV row index so I can identify the tweets in the similarity results.

In [8]:
#Build BOW Representation
#Print BOW Representation
bow_matrix = processor.build_bow(documents)

bow_words = (
    processor.bow_vectorizer.get_feature_names_out()
)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_words,
    index=tweets["CSV Row"]
)

print("BOW matrix shape:", bow_df.shape)

print("\nPreview of the first 5 tweets and 12 words:")
print(bow_df.iloc[:5, :12].to_string())

print("\nWord counts for the first tweet:")
first_bow = bow_df.iloc[0]
print(first_bow[first_bow > 0].to_string())

BOW matrix shape: (95, 707)

Preview of the first 5 tweets and 12 words:
         academic  acep  act  activity  advice  advocating  affected  aflac  agree  agriculture  alin  allotment
CSV Row                                                                                                         
0               0     0    0         0       0           0         0      0      0            0     0          0
1               0     0    0         0       0           0         0      0      0            0     0          0
5               0     0    0         0       0           0         0      0      0            0     0          0
11              0     0    0         0       0           0         0      0      0            0     1          0
12              0     0    0         0       0           0         0      0      0            0     0          0

Word counts for the first tweet:
bond                1
change              1
enjoy               1
families            1
happy         

I used CountVectorizer to build the Bag of Words representation. Each row represents a tweet, and each column represents a word. The values show how many times each word appears in a tweet. I printed a preview and the first tweet's nonzero counts to make the output easier to read.

In [9]:
#TF-IDF Representation
#Build and Print TF-IDF
tfidf_matrix = processor.build_tfidf(documents)

tfidf_words = (
    processor.tfidf_vectorizer.get_feature_names_out()
)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_words,
    index=tweets["CSV Row"]
)

print("TF-IDF matrix shape:", tfidf_df.shape)

print("\nPreview of the first 5 tweets and 12 words:")
print(tfidf_df.iloc[:5, :12].round(4).to_string())

print("\nTF-IDF weights for the first tweet:")
first_tfidf = tfidf_df.iloc[0]
print(first_tfidf[first_tfidf > 0].round(4).to_string())

TF-IDF matrix shape: (95, 707)

Preview of the first 5 tweets and 12 words:
         academic  acep  act  activity  advice  advocating  affected  aflac  agree  agriculture    alin  allotment
CSV Row                                                                                                           
0             0.0   0.0  0.0       0.0     0.0         0.0       0.0    0.0    0.0          0.0  0.0000        0.0
1             0.0   0.0  0.0       0.0     0.0         0.0       0.0    0.0    0.0          0.0  0.0000        0.0
5             0.0   0.0  0.0       0.0     0.0         0.0       0.0    0.0    0.0          0.0  0.0000        0.0
11            0.0   0.0  0.0       0.0     0.0         0.0       0.0    0.0    0.0          0.0  0.3129        0.0
12            0.0   0.0  0.0       0.0     0.0         0.0       0.0    0.0    0.0          0.0  0.0000        0.0

TF-IDF weights for the first tweet:
bond                0.2385
change              0.2781
enjoy               0.2258
f

I used TFIDF to create another representation of the tweets. Instead of just counting words, TFIDF gives them weights based on how often they appear in a tweet and across the collection. Words found in many tweets receive less emphasis. I used get_feature_names_out() to label the columns with the vocabulary words. 

In [10]:
#Calculate Cosine
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

print("\nSimilarity scores for the first 5 tweets:")
print(np.round(similarity_matrix[:5, :5], 4))

Similarity matrix shape: (95, 95)

Similarity scores for the first 5 tweets:
[[1.     0.0043 0.0094 0.0037 0.0037]
 [0.0043 1.     0.0125 0.0049 0.0049]
 [0.0094 0.0125 1.     0.0106 0.0107]
 [0.0037 0.0049 0.0106 1.     0.0042]
 [0.0037 0.0049 0.0107 0.0042 1.    ]]


I calculated cosine similarity using the TFIDF representation. This compares the tweets based on their weighted words. Higher scores mean the word patterns are more similar. The diagonal contains scores of 1 because each tweet is being compared with itself.

In [11]:
#find and print the 5 most similar pairs
#select each pair once and exclude self-comparisons
row_indices, column_indices = np.triu_indices(
    len(documents),
    k=1
)

pair_scores = similarity_matrix[
    row_indices,
    column_indices
]

#sort from highest similarity to lowest
top_positions = np.argsort(
    -pair_scores,
    kind="stable"
)[:5]

for rank, position in enumerate(top_positions, start=1):
    i = row_indices[position]
    j = column_indices[position]
    score = pair_scores[position]

    print(f"\nPair {rank}")
    print(f"Cosine similarity: {score:.4f}")

    print(f"\nTweet A — CSV row {tweets.loc[i, 'CSV Row']}:")
    print(tweets.loc[i, "Tweet Content"])

    print(f"\nTweet B — CSV row {tweets.loc[j, 'CSV Row']}:")
    print(tweets.loc[j, "Tweet Content"])

    print("\nCleaned Tweet A:")
    print(documents[i])

    print("\nCleaned Tweet B:")
    print(documents[j])


Pair 1
Cosine similarity: 1.0000

Tweet A — CSV row 159:
Whether in North South East or West Europe our Members care for animals and explain why the animal health industry is #morethanmedicine. Let AnimalhealthEurope ‘s Board member from Finland Niclas Lindstedt tell you more! https://t.co/lZpb470VjV

Tweet B — CSV row 160:
Whether in North South East or West Europe our Members care for animals and explain why the animal health industry is #morethanmedicine. Let AnimalhealthEurope ‘s Board member from Finland Niclas Lindstedt tell you more! https://t.co/lZpb470VjV

Cleaned Tweet A:
north south east west europe members care animals explain animal health industry morethanmedicine let animalhealtheurope board member finland niclas lindstedt tell

Cleaned Tweet B:
north south east west europe members care animals explain animal health industry morethanmedicine let animalhealtheurope board member finland niclas lindstedt tell

Pair 2
Cosine similarity: 0.7303

Tweet A — CSV row 317:
Today 

I printed the five tweet pairs with the highest similarity scores. I excluded self comparisons and avoided showing the same pair twice. The highest score was 1.0 for two identical tweets. The next pair scored about 0.7303 and shared wording about preventing animal disease. Printing the original text helped me understand why the tweets received high scores.

Overall in this exercise, I cleaned the Tweet Content column and kept only original tweets. I built BOW and TFIDF representations, which both contained 95 tweets and 707 word features. Then I used cosine similarity to find tweets with similar weighted vocabulary. The results included an identical pair and other tweets with related wording. This helped me see how text can be turned into numbers and compared to one another, even though similar words do not always mean the messages have the same meaning.